In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Not running in Google Colab; skipping Drive mount.")


In [ ]:
from pathlib import Path
import os


def resolve_data_path(filename: str) -> str:
    """
    Resolve data files in local Jupyter or Google Colab.
    In Colab, mount Google Drive first and place data under:
    /content/drive/MyDrive/feature-selection-experiments/data
    """
    candidates = []

    env_data_dir = os.environ.get("FEATURE_SELECTION_DATA_DIR")
    if env_data_dir:
        candidates.append(Path(env_data_dir) / filename)

    cwd = Path.cwd()
    candidates.extend([
        cwd / "data" / filename,
        cwd.parent / "data" / filename,
        Path("/content/data") / filename,
        Path("/content/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/feature-selection-experiments/data") / filename,
        Path("/content/drive/MyDrive/data") / filename,
        Path(r"D:\feature-selection-experiments\data") / filename,
    ])

    for path in candidates:
        if path.exists():
            return str(path)

    search_roots = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ]
    for root in search_roots:
        if not root.exists():
            continue
        matches = list(root.glob(f"**/{filename}"))
        for match in matches:
            if match.parent.name == "data":
                return str(match)
        if matches:
            return str(matches[0])

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Could not find {filename}. Checked:\n{checked}\n\n"
        "Upload the data folder to /content/data, or mount Google Drive and place it at "
        "/content/drive/MyDrive/feature-selection-experiments/data."
    )


DATA_DIR = Path(resolve_data_path("X_train.csv")).parent
if Path("/content/drive/MyDrive").exists():
    RESULT_DIR = Path("/content/drive/MyDrive/feature-selection-experiments/results/de-qlearning")
else:
    RESULT_DIR = Path.cwd() / "results" / "de-qlearning"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = RESULT_DIR


def output_path(filename: str) -> str:
    return str(RESULT_DIR / filename)


print(f"Using DATA_DIR: {DATA_DIR}")
print(f"Using RESULT_DIR: {RESULT_DIR}")


In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Tuple, List, Dict, Optional

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC

In [2]:
@dataclass
class QLDEConfig:
    population_size: int = 20
    max_generations: int = 30
    crossover_rate: float = 0.9
    init_F_min: float = 0.3
    init_F_max: float = 0.8
    q_learning_rate: float = 0.1   # alpha
    q_discount: float = 0.9        # gamma
    softmax_temperature: float = 1.0
    random_state: int = 42
    cv_splits: int = 5

    # Fitness = alpha_metric * macro_f1 - beta_size * feature_ratio
    alpha_metric: float = 0.95
    beta_size: float = 0.05

    # Các action điều chỉnh F
    actions: Tuple[float, ...] = (-0.05, 0.0, 0.05)

    # Ngưỡng biến continuous -> binary mask
    threshold: float = 0.5

    # Giới hạn F
    F_lower_bound: float = 0.1
    F_upper_bound: float = 1.0

    # Nếu cá thể chọn ít hơn min_features thì tự sửa
    min_features: int = 1

    # Dùng top ratio để cluster-guided mutation như paper
    top_ratio: float = 0.5
    val_size: float = 0.2

In [3]:
def load_data(
    x_path: str,
    y_path: str
) -> Tuple[pd.DataFrame, np.ndarray]:
    X = pd.read_csv(x_path)
    y_df = pd.read_csv(y_path)

    if y_df.shape[1] != 1:
        raise ValueError("y_train.csv phải chỉ có đúng 1 cột nhãn.")

    y = y_df.iloc[:, 0].values

    # Encode nhãn nếu là chuỗi
    if y.dtype == object or str(y.dtype).startswith("string"):
        le = LabelEncoder()
        y = le.fit_transform(y)

    return X, y

In [4]:
def logistic_map_init(
    population_size: int,
    n_features: int,
    rng: np.random.Generator,
    mu: float = 3.99
) -> np.ndarray:
    """
    Khởi tạo quần thể bằng Logistic chaotic mapping.
    """
    pop = np.zeros((population_size, n_features), dtype=float)

    for i in range(population_size):
        phi = rng.uniform(0.001, 0.999)
        for j in range(n_features):
            phi = mu * phi * (1.0 - phi)
            pop[i, j] = phi

    return pop


def continuous_to_mask(
    vector: np.ndarray,
    threshold: float,
    min_features: int,
    rng: np.random.Generator
) -> np.ndarray:
    """
    Convert continuous vector [0,1] -> binary mask.
    """
    mask = (vector > threshold).astype(int)

    if mask.sum() < min_features:
        # ép chọn ngẫu nhiên min_features cột
        idx = rng.choice(len(mask), size=min_features, replace=False)
        mask[idx] = 1

    return mask



def hamming_distance_ratio(mask_a: np.ndarray, mask_b: np.ndarray) -> float:
    """Return the normalized Hamming distance between two binary masks."""
    mask_a = np.asarray(mask_a).astype(int)
    mask_b = np.asarray(mask_b).astype(int)
    if mask_a.shape != mask_b.shape:
        raise ValueError("Two masks must have the same shape.")
    return float(np.mean(mask_a != mask_b))


def continuous_population_to_masks(
    population: np.ndarray,
    threshold: float = 0.5,
    min_features: int = 1,
    use_sigmoid: bool = False
) -> np.ndarray:
    """Convert a continuous population to binary feature masks for metric tracking."""
    values = 1.0 / (1.0 + np.exp(-population)) if use_sigmoid else np.asarray(population)
    masks = (values > threshold).astype(int)

    if min_features > 0:
        for i in range(masks.shape[0]):
            if masks[i].sum() < min_features:
                top_idx = np.argsort(values[i])[-min_features:]
                masks[i, top_idx] = 1

    return masks


def compute_population_diversity(population_masks: np.ndarray) -> float:
    """Average pairwise Hamming distance among all masks in the current population."""
    population_masks = np.asarray(population_masks).astype(int)
    n_individuals = population_masks.shape[0]
    if n_individuals < 2:
        return 0.0

    total_distance = 0.0
    pair_count = 0
    for i in range(n_individuals):
        for j in range(i + 1, n_individuals):
            total_distance += hamming_distance_ratio(population_masks[i], population_masks[j])
            pair_count += 1

    return float(total_distance / pair_count)


def compute_exploration_exploitation(
    new_masks: np.ndarray,
    previous_masks: np.ndarray,
    similarity_threshold: float = 0.1
) -> Tuple[float, float]:
    """
    Exploration is the fraction of new masks whose nearest previous neighbor
    is farther than the threshold. Exploitation is the complementary fraction.
    """
    new_masks = np.asarray(new_masks).astype(int)
    previous_masks = np.asarray(previous_masks).astype(int)
    if len(new_masks) == 0 or len(previous_masks) == 0:
        return 0.0, 0.0

    exploration_count = 0
    for new_mask in new_masks:
        nearest_distance = min(
            hamming_distance_ratio(new_mask, previous_mask)
            for previous_mask in previous_masks
        )
        if nearest_distance > similarity_threshold:
            exploration_count += 1

    exploration = exploration_count / len(new_masks)
    exploitation = 1.0 - exploration
    return float(exploration), float(exploitation)

def select_action_softmax(
    q_table: np.ndarray,
    state: int,
    temperature: float,
    rng: np.random.Generator
) -> int:
    """
    Chọn action theo softmax trên hàng Q[state].
    state: 0 hoặc 1
    """
    q_values = q_table[state].copy()

    if temperature <= 0:
        return int(np.argmax(q_values))

    q_scaled = q_values / temperature
    q_scaled = q_scaled - np.max(q_scaled)
    probs = np.exp(q_scaled)
    probs = probs / probs.sum()

    return int(rng.choice(len(q_values), p=probs))


def evaluate_feature_subset(
    X: pd.DataFrame,
    y: np.ndarray,
    mask: np.ndarray,
    config: QLDEConfig
) -> Tuple[float, float, int]:
    """
    Evaluate một tập feature bằng 1 lần chia train/validation.
    Trả về:
    - fitness
    - macro_f1 trên validation
    - số lượng feature được chọn
    """
    selected_cols = X.columns[mask == 1]
    X_sel = X[selected_cols]

    if X_sel.shape[1] == 0:
        return -1e9, 0.0, 0

    X_train_fs, X_val_fs, y_train_fs, y_val_fs = train_test_split(
        X_sel,
        y,
        test_size=config.val_size,
        random_state=config.random_state,
        stratify=y
    )

    clf = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=config.random_state)

    clf.fit(X_train_fs, y_train_fs)
    y_pred = clf.predict(X_val_fs)

    macro_f1 = f1_score(y_val_fs, y_pred, average="macro")
    feature_ratio = X_sel.shape[1] / X.shape[1]

    fitness = (
        config.alpha_metric * macro_f1
        - config.beta_size * feature_ratio
    )

    return fitness, macro_f1, X_sel.shape[1]


class QLDEFeatureSelector:
    def __init__(self, config: Optional[QLDEConfig] = None):
        self.config = config if config is not None else QLDEConfig()
        self.rng = np.random.default_rng(self.config.random_state)

        self.best_mask_: Optional[np.ndarray] = None
        self.best_fitness_: float = -np.inf
        self.best_macro_f1_: float = 0.0
        self.best_num_features_: int = 0
        self.history_: List[Dict] = []
        self.optimization_metrics_: Optional[pd.DataFrame] = None
        self.q_table_: Optional[np.ndarray] = None

    def _initialize_population(self, n_features: int) -> Tuple[np.ndarray, np.ndarray]:
        population = logistic_map_init(
            population_size=self.config.population_size,
            n_features=n_features,
            rng=self.rng
        )

        # F riêng cho từng cá thể như paper
        F_values = self.rng.uniform(
            self.config.init_F_min,
            self.config.init_F_max,
            size=self.config.population_size
        )

        return population, F_values

    def fit(self, X: pd.DataFrame, y: np.ndarray):
        n_features = X.shape[1]

        # Q-table: 2 states x num_actions
        # state 0 = cải thiện, state 1 = không cải thiện
        q_table = np.zeros((2, len(self.config.actions)), dtype=float)

        population, F_values = self._initialize_population(n_features)

        fitnesses = np.zeros(self.config.population_size)
        macro_f1s = np.zeros(self.config.population_size)
        num_features = np.zeros(self.config.population_size, dtype=int)

        # Đánh giá quần thể ban đầu
        for i in range(self.config.population_size):
            mask = continuous_to_mask(
                population[i],
                self.config.threshold,
                self.config.min_features,
                self.rng
            )
            fit_i, f1_i, nf_i = evaluate_feature_subset(X, y, mask, self.config)
            fitnesses[i] = fit_i
            macro_f1s[i] = f1_i
            num_features[i] = nf_i

            if fit_i > self.best_fitness_:
                self.best_fitness_ = fit_i
                self.best_mask_ = mask.copy()
                self.best_macro_f1_ = f1_i
                self.best_num_features_ = nf_i

        for gen in range(self.config.max_generations):
            previous_masks_for_metrics = continuous_population_to_masks(
                population,
                threshold=self.config.threshold,
                min_features=self.config.min_features
            )
            trial_masks_for_metrics = np.zeros_like(previous_masks_for_metrics)

            new_population = population.copy()
            new_fitnesses = fitnesses.copy()
            new_macro_f1s = macro_f1s.copy()
            new_num_features = num_features.copy()

            # top 50% cá thể tốt hơn để cluster-guided mutation
            sorted_idx = np.argsort(-fitnesses)
            top_k = max(1, int(self.config.top_ratio * self.config.population_size))
            elite_indices = sorted_idx[:top_k]

            gen_improvements = 0

            for i in range(self.config.population_size):
                parent = population[i].copy()
                parent_fitness = fitnesses[i]

                # Xác định state hiện tại từ vòng trước
                # nếu chưa có thông tin, cho mặc định state = không cải thiện
                current_state = 1

                # Chọn action theo softmax
                action_idx = select_action_softmax(
                    q_table=q_table,
                    state=current_state,
                    temperature=self.config.softmax_temperature,
                    rng=self.rng
                )
                delta_F = self.config.actions[action_idx]

                # Update F_i
                F_i = F_values[i] + delta_F
                F_i = float(np.clip(
                    F_i,
                    self.config.F_lower_bound,
                    self.config.F_upper_bound
                ))

                # Cluster-guided mutation
                # base vector lấy từ top 50%
                Lk = int(self.rng.choice(elite_indices))

                # chọn r1, r2 khác nhau và khác i
                candidates = [idx for idx in range(self.config.population_size) if idx != i]
                r1, r2 = self.rng.choice(candidates, size=2, replace=False)

                mutant = population[Lk] + F_i * (population[r1] - population[r2])
                mutant = np.clip(mutant, 0.0, 1.0)

                # Binomial crossover
                trial = parent.copy()
                j_rand = self.rng.integers(0, n_features)

                for j in range(n_features):
                    if self.rng.random() <= self.config.crossover_rate or j == j_rand:
                        trial[j] = mutant[j]

                trial = np.clip(trial, 0.0, 1.0)

                # Evaluate trial
                trial_mask = continuous_to_mask(
                    trial,
                    self.config.threshold,
                    self.config.min_features,
                    self.rng
                )
                trial_fitness, trial_f1, trial_nf = evaluate_feature_subset(
                    X, y, trial_mask, self.config
                )
                trial_masks_for_metrics[i] = trial_mask

                # Reward + next state
                if trial_fitness > parent_fitness:
                    reward = 1.0
                    next_state = 0  # cải thiện
                else:
                    reward = 0.0
                    next_state = 1  # không cải thiện

                # Q update
                best_next_q = np.max(q_table[next_state])
                old_q = q_table[current_state, action_idx]
                q_table[current_state, action_idx] = old_q + self.config.q_learning_rate * (
                    reward + self.config.q_discount * best_next_q - old_q
                )

                # Selection
                if trial_fitness > parent_fitness:
                    new_population[i] = trial
                    new_fitnesses[i] = trial_fitness
                    new_macro_f1s[i] = trial_f1
                    new_num_features[i] = trial_nf
                    F_values[i] = F_i
                    gen_improvements += 1

                    if trial_fitness > self.best_fitness_:
                        self.best_fitness_ = trial_fitness
                        self.best_mask_ = trial_mask.copy()
                        self.best_macro_f1_ = trial_f1
                        self.best_num_features_ = trial_nf

            population = new_population
            fitnesses = new_fitnesses
            macro_f1s = new_macro_f1s
            num_features = new_num_features

            current_masks_for_metrics = continuous_population_to_masks(
                population,
                threshold=self.config.threshold,
                min_features=self.config.min_features
            )
            diversity = compute_population_diversity(current_masks_for_metrics)
            exploration_threshold = getattr(self.config, "exploration_threshold", 0.1)
            exploration, exploitation = compute_exploration_exploitation(
                new_masks=trial_masks_for_metrics,
                previous_masks=previous_masks_for_metrics,
                similarity_threshold=exploration_threshold
            )

            self.history_.append({
                "generation": gen + 1,
                "best_fitness": float(np.max(fitnesses)),
                "mean_fitness": float(np.mean(fitnesses)),
                "best_macro_f1": float(np.max(macro_f1s)),
                "mean_macro_f1": float(np.mean(macro_f1s)),
                "best_num_features": int(num_features[np.argmax(fitnesses)]),
                "improvements": gen_improvements,
                "diversity": diversity,
                "exploration": exploration,
                "exploitation": exploitation,
                "exploration_threshold": exploration_threshold
            })

            print(
                f"Generation {gen+1:02d} | "
                f"Best fitness = {np.max(fitnesses):.5f} | "
                f"Best macro-F1 = {np.max(macro_f1s):.5f} | "
                f"Selected features = {num_features[np.argmax(fitnesses)]} | "
                f"Improvements = {gen_improvements} | "
                f"Diversity = {diversity:.4f} | "
                f"Exploration = {exploration:.4f} | "
                f"Exploitation = {exploitation:.4f}"
            )

        self.optimization_metrics_ = pd.DataFrame(self.history_)
        self.optimization_metrics_.to_csv(output_path("optimization_metrics_de_qlearning_with_formulas.csv"), index=False)
        print("[Metrics] Saved optimization_metrics_de_qlearning_with_formulas.csv")
        self.q_table_ = q_table
        return self

    def get_selected_features(self, feature_names: List[str]) -> List[str]:
        if self.best_mask_ is None:
            raise ValueError("Model chưa fit.")
        return [f for f, m in zip(feature_names, self.best_mask_) if m == 1]

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        if self.best_mask_ is None:
            raise ValueError("Model chưa fit.")
        selected_cols = X.columns[self.best_mask_ == 1]
        return X[selected_cols]

    def fit_transform(self, X: pd.DataFrame, y: np.ndarray) -> pd.DataFrame:
        self.fit(X, y)
        return self.transform(X)

In [5]:
if __name__ == "__main__":
    # Đổi lại đúng tên file của bạn
    X_path = resolve_data_path("X_train.csv")
    y_path = resolve_data_path("y_train.csv")

    X, y = load_data(X_path, y_path)

    config = QLDEConfig(
        population_size=20,
        max_generations=50,
        crossover_rate=0.9,
        init_F_min=0.3,
        init_F_max=0.8,
        q_learning_rate=0.1,
        q_discount=0.9,
        softmax_temperature=1.0,
        cv_splits=5,
        alpha_metric=0.95,
        beta_size=0.05,
        actions=(-0.05, 0.0, 0.05),
        threshold=0.5,
        min_features=1,
        random_state=42
    )

    selector = QLDEFeatureSelector(config=config)
    selector.fit(X, y)

    selected_features = selector.get_selected_features(list(X.columns))
    X_selected = selector.transform(X)

    print("\n========== KẾT QUẢ CUỐI ==========")
    print(f"Best fitness       : {selector.best_fitness_:.6f}")
    print(f"Best macro-F1      : {selector.best_macro_f1_:.6f}")
    print(f"Số feature đã chọn : {selector.best_num_features_}")
    print(f"Tên feature chọn   : {selected_features}")

    print("\nQ-table cuối:")
    print(selector.q_table_)

    # Lưu file kết quả
    X_selected.to_csv(output_path("X_train_selected.csv"), index=False)
    pd.DataFrame({"selected_feature": selected_features}).to_csv(
        output_path("selected_features_list.csv"), index=False
    )
    pd.DataFrame(selector.history_).to_csv(output_path("qlde_history.csv"), index=False)

    print("\nĐã lưu:")
    print("- X_train_selected.csv")
    print("- selected_features_list.csv")
    print("- qlde_history.csv")

Generation 01 | Best fitness = 0.68031 | Best macro-F1 = 0.74391 | Selected features = 528 | Improvements = 10
Generation 02 | Best fitness = 0.68426 | Best macro-F1 = 0.74906 | Selected features = 547 | Improvements = 13
Generation 03 | Best fitness = 0.69234 | Best macro-F1 = 0.75630 | Selected features = 523 | Improvements = 10
Generation 04 | Best fitness = 0.69234 | Best macro-F1 = 0.75630 | Selected features = 523 | Improvements = 10
Generation 05 | Best fitness = 0.69562 | Best macro-F1 = 0.76018 | Selected features = 531 | Improvements = 6
Generation 06 | Best fitness = 0.69562 | Best macro-F1 = 0.76018 | Selected features = 531 | Improvements = 6
Generation 07 | Best fitness = 0.69755 | Best macro-F1 = 0.76236 | Selected features = 534 | Improvements = 8
Generation 08 | Best fitness = 0.69755 | Best macro-F1 = 0.76236 | Selected features = 534 | Improvements = 7
Generation 09 | Best fitness = 0.70603 | Best macro-F1 = 0.77140 | Selected features = 536 | Improvements = 4
Genera

In [6]:
import pandas as pd

# Đọc dữ liệu
X_train = pd.read_csv(resolve_data_path("X_train.csv"))
X_val = pd.read_csv(resolve_data_path("X_val.csv"))
X_test = pd.read_csv(resolve_data_path("X_test.csv"))

y_train = pd.read_csv(resolve_data_path("y_train.csv")).iloc[:, 0].values
y_val = pd.read_csv(resolve_data_path("y_val.csv")).iloc[:, 0].values
y_test = pd.read_csv(resolve_data_path("y_test.csv")).iloc[:, 0].values

# Áp feature selected lên cả 3 tập
X_train_selected = X_train[selected_features].copy()
X_val_selected = X_val[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

print("Train selected shape:", X_train_selected.shape)
print("Val selected shape:", X_val_selected.shape)
print("Test selected shape:", X_test_selected.shape)

Train selected shape: (16000, 551)
Val selected shape: (2000, 551)
Test selected shape: (2000, 551)


In [7]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score

def compute_metrics(y_true, y_pred, model_name):
    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   output_dict=True, zero_division=0)
    row = {"model": model_name}
    for cls in CLASS_NAMES:
        for m in ["precision","recall","f1-score"]:
            col = f"{cls}_{m.replace('-score','')}"
            row[col] = round(report[cls][m], 4)
    for avg in ["macro avg","weighted avg"]:
        prefix = avg.replace(" avg","")
        for m in ["precision","recall","f1-score"]:
            col = f"{prefix}_{m.replace('-score','')}"
            row[col] = round(report[avg][m], 4)
    row["accuracy"] = round(report["accuracy"], 4)
    return row

In [10]:
all_results = [] 

from sklearn.neighbors         import KNeighborsClassifier
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.naive_bayes       import GaussianNB
from sklearn.linear_model      import LogisticRegression
from sklearn.svm               import LinearSVC, SVC
from sklearn.model_selection   import GridSearchCV
from sklearn.pipeline          import Pipeline
import numpy as np

unique_classes = np.unique(y_train)
N_CLASSES = len(unique_classes)
CLASS_NAMES = [str(c) for c in unique_classes]

print("N_CLASSES :", N_CLASSES)
print("CLASS_NAMES:", CLASS_NAMES)

ML_CONFIGS = [
    (
        "KNN",
        KNeighborsClassifier(),
        {"n_neighbors":[3,5,7,11], "weights":["uniform","distance"],
         "metric":["euclidean","manhattan"]}
    ),
    (
        "DecisionTree",
        DecisionTreeClassifier(random_state=42),
        {"max_depth":[None,10,20,30], "min_samples_split":[2,5,10],
         "criterion":["gini","entropy"]}
    ),
    (
        "RandomForest",
        RandomForestClassifier(random_state=42, n_jobs=-1),
        {"n_estimators":[100,200], "max_depth":[None,10,20],
         "min_samples_split":[2,5]}
    ),
    (
        "GaussianNB",
        GaussianNB(),
        {"var_smoothing":[1e-9,1e-8,1e-7,1e-6]}
    ),
    (
        "LogisticRegression",
        LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1),
        {"C":[0.01,0.1,1,10], "solver":["lbfgs","saga"],
         "multi_class":["multinomial"]}
    ),
    (
        "LinearSVM",
        LinearSVC(max_iter=2000, random_state=42),
        {"C":[0.01,0.1,1,10]}
    ),
    (
        "RBF_SVM",
        SVC(kernel="rbf", random_state=42),
        {"C":[0.1,1,10], "gamma":["scale","auto"]}
    ),
]

import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report

all_results = []
report_rows = []

for name, estimator, param_grid in ML_CONFIGS:
    print(f"\n[{name}] GridSearch đang chạy...")

    gs = GridSearchCV(
        estimator,
        param_grid,
        cv=5,
        scoring="f1_macro",
        n_jobs=-1,
        verbose=0
    )

    gs.fit(X_train_selected, y_train)

    best = gs.best_estimator_
    y_pred = best.predict(X_test_selected)

    print(f"  Best params : {gs.best_params_}")
    print(f"  CV F1-macro : {gs.best_score_:.4f}")

    # report dạng dict thay vì in text thô
    report_dict = classification_report(
        y_test,
        y_pred,
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True
    )

    # Gom toàn bộ report của model thành 1 hàng
    flat_row = {"model": name}

    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):  # các class, macro avg, weighted avg
            for metric_name, value in metrics.items():
                flat_row[f"{label}_{metric_name}"] = value
        else:  # accuracy là scalar
            flat_row[label] = metrics

    flat_row["best_params"] = str(gs.best_params_)
    flat_row["cv_f1_macro"] = gs.best_score_
    report_rows.append(flat_row)

    # phần cũ của bạn nếu vẫn muốn giữ
    row = compute_metrics(y_test, y_pred, name)
    row["best_params"] = str(gs.best_params_)
    all_results.append(row)

# Bảng classification report: mỗi hàng là 1 model
classification_report_df = pd.DataFrame(report_rows)

# Sắp xếp cột cho dễ nhìn
first_cols = ["model", "accuracy", "cv_f1_macro", "best_params"]
other_cols = [c for c in classification_report_df.columns if c not in first_cols]
classification_report_df = classification_report_df[first_cols + other_cols]

display(classification_report_df)

pd.DataFrame(all_results).to_csv(
    output_path("all_results_models.csv"),
    index=False,
    encoding="utf-8-sig"
)

N_CLASSES : 8
CLASS_NAMES: ['0', '1', '2', '3', '4', '5', '6', '7']

[KNN] GridSearch đang chạy...
  Best params : {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
  CV F1-macro : 0.5582

[DecisionTree] GridSearch đang chạy...
  Best params : {'criterion': 'gini', 'max_depth': 10, 'min_samples_split': 5}
  CV F1-macro : 0.6214

[RandomForest] GridSearch đang chạy...
  Best params : {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
  CV F1-macro : 0.7732

[GaussianNB] GridSearch đang chạy...
  Best params : {'var_smoothing': 1e-06}
  CV F1-macro : 0.3548

[LogisticRegression] GridSearch đang chạy...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and wi

  Best params : {'C': 0.01, 'multi_class': 'multinomial', 'solver': 'lbfgs'}
  CV F1-macro : 0.7789

[LinearSVM] GridSearch đang chạy...
  Best params : {'C': 0.01}
  CV F1-macro : 0.7761

[RBF_SVM] GridSearch đang chạy...
  Best params : {'C': 10, 'gamma': 'auto'}
  CV F1-macro : 0.8400


,model,accuracy,cv_f1_macro,best_params,0_precision,0_recall,0_f1-score,0_support,1_precision,1_recall,...,7_f1-score,7_support,macro avg_precision,macro avg_recall,macro avg_f1-score,macro avg_support,weighted avg_precision,weighted avg_recall,weighted avg_f1-score,weighted avg_support
0,KNN,0.5715,0.558178,"{'metric': 'manhattan', 'n_neighbors': 11, 'we...",0.587500,0.188,0.284848,250.0,0.412758,0.880,...,0.686916,250.0,0.611974,0.5715,0.561500,2000.0,0.611974,0.5715,0.561500,2000.0
1,DecisionTree,0.6345,0.621444,"{'criterion': 'gini', 'max_depth': 10, 'min_sa...",0.451163,0.388,0.417204,250.0,0.739300,0.760,...,0.610487,250.0,0.634412,0.6345,0.633039,2000.0,0.634412,0.6345,0.633039,2000.0
2,RandomForest,0.7885,0.773192,"{'max_depth': None, 'min_samples_split': 5, 'n...",0.836158,0.592,0.693208,250.0,0.782456,0.892,...,0.803419,250.0,0.799437,0.7885,0.787254,2000.0,0.799437,0.7885,0.787254,2000.0
3,GaussianNB,0.3640,0.354767,{'var_smoothing': 1e-06},0.220779,0.068,0.103976,250.0,0.358306,0.880,...,0.492228,250.0,0.400093,0.3640,0.344828,2000.0,0.400093,0.3640,0.344828,2000.0
4,LogisticRegression,0.7970,0.778886,"{'C': 0.01, 'multi_class': 'multinomial', 'sol...",0.713080,0.676,0.694045,250.0,0.819495,0.908,...,0.827160,250.0,0.796070,0.7970,0.795965,2000.0,0.796070,0.7970,0.795965,2000.0
5,LinearSVM,0.8035,0.776067,{'C': 0.01},0.775120,0.648,0.705882,250.0,0.810714,0.908,...,0.836364,250.0,0.802581,0.8035,0.801620,2000.0,0.802581,0.8035,0.801620,2000.0
6,RBF_SVM,0.8490,0.839966,"{'C': 10, 'gamma': 'auto'}",0.780591,0.740,0.759754,250.0,0.861004,0.892,...,0.859470,250.0,0.848629,0.8490,0.848522,2000.0,0.848629,0.8490,0.848522,2000.0


In [ ]:
N_FEATURES = X_train.shape[1]
SAVE_DIR   = str(RESULT_DIR)

In [ ]:
X_train_dl = X_train.values.reshape(-1, 1, N_FEATURES)
X_val_dl   = X_val.values.reshape(-1, 1, N_FEATURES)
X_test_dl  = X_test.values.reshape(-1, 1, N_FEATURES)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import (ModelCheckpoint, EarlyStopping,
                                        ReduceLROnPlateau, CSVLogger)
from tensorflow.keras.utils import to_categorical
import os

tf.random.set_seed(42)

y_train_oh = to_categorical(y_train, N_CLASSES)
y_val_oh   = to_categorical(y_val,   N_CLASSES)
y_test_oh  = to_categorical(y_test,  N_CLASSES)

EPOCHS     = 100
BATCH_SIZE = 64
PATIENCE   = 15

def dl_callbacks(model_name):
    ckpt_path = f"{SAVE_DIR}/{model_name}_best.keras"
    return [
        ModelCheckpoint(ckpt_path, monitor="val_f1_macro",
                        mode="max", save_best_only=True, verbose=0),
        EarlyStopping(monitor="val_f1_macro", mode="max",
                      patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=7, verbose=1),
        CSVLogger(f"{SAVE_DIR}/{model_name}_history.csv"),
    ]

# Custom F1 metric (macro) để monitor trong Keras
import tensorflow.keras.backend as K

class MacroF1(keras.metrics.Metric):
    def __init__(self, n_classes, name="f1_macro", **kw):
        super().__init__(name=name, **kw)
        self.n_classes = n_classes
        self.f1 = self.add_weight(name="f1", initializer="zeros")
        self.count = self.add_weight(name="count", initializer="zeros")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(tf.argmax(y_true, axis=-1), tf.int32)
        y_pred = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
        f1_sum = tf.constant(0.0)
        for c in range(self.n_classes):
            tp = tf.reduce_sum(tf.cast((y_true==c) & (y_pred==c), tf.float32))
            fp = tf.reduce_sum(tf.cast((y_true!=c) & (y_pred==c), tf.float32))
            fn = tf.reduce_sum(tf.cast((y_true==c) & (y_pred!=c), tf.float32))
            p  = tp / (tp + fp + 1e-8)
            r  = tp / (tp + fn + 1e-8)
            f1_sum += 2*p*r/(p+r+1e-8)
        self.f1.assign(f1_sum / self.n_classes)
        self.count.assign_add(1.0)

    def result(self):
        return self.f1

    def reset_state(self):
        self.f1.assign(0.0)
        self.count.assign(0.0)

def compile_model(model):
    model.compile(
        optimizer=keras.optimizers.Adam(1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy", MacroF1(N_CLASSES)]
    )
    return model

def eval_dl(model_name, model):
    """Load best weights → predict → classification report."""
    ckpt = f"{SAVE_DIR}/{model_name}_best.keras"
    if os.path.exists(ckpt):
        model.load_weights(ckpt)
    y_prob = model.predict(X_test_dl, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    print(f"\n[{model_name}] — Test Classification Report:")
    print(classification_report(y_test, y_pred,
                                target_names=CLASS_NAMES, zero_division=0))
    row = compute_metrics(y_test, y_pred, model_name)
    all_results.append(row)

def fit_dl(model_name, model):
    compile_model(model)
    history = model.fit(
        X_train_dl, y_train_oh,
        validation_data=(X_val_dl, y_val_oh),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=dl_callbacks(model_name),
        verbose=1
    )
    eval_dl(model_name, model)
    return history

print("\n" + "="*60)
print("DEEP LEARNING MODELS")
print("="*60)

def build_mlp():
    inp = keras.Input(shape=(N_FEATURES,))
    x   = layers.Dense(256, activation="relu")(inp)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(128, activation="relu")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="MLP")
    
def build_lstm():
    inp = keras.Input(shape=(1, N_FEATURES))
    x   = layers.LSTM(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.LSTM(64)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="LSTM")

fit_dl("LSTM", build_lstm())

# ── B2. BiLSTM ──────────────────────────────────────────────
def build_bilstm():
    inp = keras.Input(shape=(1, N_FEATURES))
    x   = layers.Bidirectional(layers.LSTM(128, return_sequences=True))(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Bidirectional(layers.LSTM(64))(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="BiLSTM")

fit_dl("BiLSTM", build_bilstm())

# ── B3. GRU ─────────────────────────────────────────────────
def build_gru():
    inp = keras.Input(shape=(1, N_FEATURES))
    x   = layers.GRU(128, return_sequences=True)(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.GRU(64)(x)
    x   = layers.Dropout(0.3)(x)
    x   = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="GRU")

fit_dl("GRU", build_gru())

# ── B4. CNN1D ────────────────────────────────────────────────
def build_cnn1d():
    inp = keras.Input(shape=(1, N_FEATURES))
    # treat features as "channels" → transpose để Conv1D hợp lý
    x   = layers.Permute((2, 1))(inp)          # (batch, features, 1)
    x   = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
    x   = layers.BatchNormalization()(x)
    x   = layers.GlobalMaxPooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="CNN1D")

fit_dl("CNN1D", build_cnn1d())

# ── B5. Transformer ─────────────────────────────────────────
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    # Multi-head attention
    x = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=head_size, dropout=dropout
        )(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + inputs)
    # Feed-forward
    ff = layers.Dense(ff_dim, activation="relu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(inputs.shape[-1])(ff)
    return layers.LayerNormalization(epsilon=1e-6)(ff + x)

def build_transformer():
    inp = keras.Input(shape=(1, N_FEATURES))
    x   = transformer_encoder(inp, head_size=32, num_heads=4,
                               ff_dim=128, dropout=0.1)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(64, activation="relu")(x)
    x   = layers.Dropout(0.2)(x)
    out = layers.Dense(N_CLASSES, activation="softmax")(x)
    return keras.Model(inp, out, name="Transformer")

fit_dl("Transformer", build_transformer())

# ════════════════════════════════════════════════════════════
# PHẦN C — XUẤT KẾT QUẢ CSV
# ════════════════════════════════════════════════════════════
results_df = pd.DataFrame(all_results)

# Đảm bảo cột "model" đứng đầu
cols = ["model"] + [c for c in results_df.columns if c != "model"]
results_df = results_df[cols]

out_path = output_path("model_comparison.csv")
results_df.to_csv(out_path, index=False)

print("\n" + "="*60)
print("KẾT QUẢ TỔNG HỢP")
print("="*60)
print(results_df[["model","accuracy","macro_precision",
                   "macro_recall","macro_f1"]].to_string(index=False))
print(f"\nĐã lưu full CSV: {out_path}")
print(f"Best models DL: {SAVE_DIR}/")